# WBT-SOHO Phase 1D - CIFAR-100 train-only gate

This notebook tests transient empirical-residual transport and WTA-aware boundary coverage. It uses nested splits of CIFAR-100 **training data only**. It must never restore, extract, or open `test.pt`.

In [ ]:
# === Edit only repository/path values in this cell. Do not edit the locked protocol. ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'experiment/soho-selfcontained'
WORK_DIR = '/content/SOHO-CL'
DATASET_KEY = 'cifar100'
FEATURE_CACHE_ROOT = '/content/wbt_soho_phase1d_features'
OUTPUT_ROOT = '/content/wbt_soho_phase1d_outputs'
# Optional existing directory/archive containing metadata.json + train.pt only.
TRAIN_CACHE_SOURCE = ''
BATCH_SIZE = 128
NUM_WORKERS = 2
EXPECTED_CONFIG_SHA256 = '82b98079a6e30a81e968b3da24d38758bdabcb7d6637580700e6e024cc355e7a'
EXPECTED_RUNNER_SHA256 = '475b7251703d15e00ff2b32378674869af612254331656e9f5a1784c6a5d2b1b'
EXPECTED_TRANSPORT_SHA256 = 'c1cc367ec62fd10eddb8acad3cc3c92c5ad2517a7804d8b78c4950e36dea73c3'
EXPECTED_LEARNER_SHA256 = '74de0ad87f2c9f899542caccc2eb24ccfd4e911c8608b80d98bf43a34ff520ea'

In [ ]:
# Fresh checkout, dependencies and immutable source verification.
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
os.chdir('/content')
repo = Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR],check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub','pandas','matplotlib','seaborn'],check=True)
import torch
assert torch.cuda.is_available(), 'Select Runtime -> Change runtime type -> T4 GPU.'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
CONFIG = 'configs/wbt_soho_phase1d_cifar100_train_only.json'
RUNNER = 'tools/wbt_soho_phase1d.py'
assert sha(CONFIG) == EXPECTED_CONFIG_SHA256, 'Config hash mismatch'
assert sha(RUNNER) == EXPECTED_RUNNER_SHA256, 'Runner hash mismatch'
assert sha('methods/wbt_soho/transport.py') == EXPECTED_TRANSPORT_SHA256
assert sha('methods/wbt_soho/learner.py') == EXPECTED_LEARNER_SHA256
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
print('GPU:',torch.cuda.get_device_name(0))
print('commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('WBT-SOHO SOURCE CHECK: PASS')

In [ ]:
# Synthetic mathematical, learner-state and resumable-runner gate.
tests=['tests/test_wbt_soho_math.py','tests/test_wbt_soho_learner.py','tests/test_wbt_soho_phase1d.py']
completed=subprocess.run([sys.executable,'-B','-m','pytest','-q',*tests],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
print(completed.stdout,flush=True)
assert completed.returncode == 0, 'Correctness gate failed.'
print('WBT-SOHO CORRECTNESS GATE: PASS')

In [ ]:
# Restore a TRAIN-only feature cache when available; otherwise extract it once.
protocol=json.loads(Path(CONFIG).read_text())
dataset=protocol['datasets'][DATASET_KEY]
cache=Path(FEATURE_CACHE_ROOT)/DATASET_KEY
candidate_sources=[]
if TRAIN_CACHE_SOURCE: candidate_sources.append(Path(TRAIN_CACHE_SOURCE))
candidate_sources += [Path('/content/tsoho_cifar100_cache'),Path('/content/mars_soho_phase1_features/cifar100')]
source=next((p for p in candidate_sources if p.exists()),None)
if not (cache/'train.pt').is_file() and source is not None:
    cache.mkdir(parents=True,exist_ok=True)
    if source.is_dir():
        for name in ('metadata.json','train.pt'):
            assert (source/name).is_file(),f'Missing {name} in {source}'
            shutil.copy2(source/name,cache/name)
    else:
        assert zipfile.is_zipfile(source),f'Not a ZIP: {source}'
        with zipfile.ZipFile(source) as archive:
            matches={Path(name).name:name for name in archive.namelist() if Path(name).name in {'metadata.json','train.pt'}}
            assert set(matches)=={'metadata.json','train.pt'}
            for name,member in matches.items(): (cache/name).write_bytes(archive.read(member))
if not (cache/'train.pt').is_file():
    import kagglehub
    from huggingface_hub import hf_hub_download
    checkpoint=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
    assert Path(checkpoint).stat().st_size==346284714 and sha(checkpoint)==protocol['backbone']['checkpoint_sha256']
    dataset_root=kagglehub.dataset_download('zaphat206/cifar-100')
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',dataset_root,'--backbone-checkpoint',checkpoint,'--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256',protocol['backbone']['checkpoint_sha256'],'--feature-cache-dir',str(cache),'--output-dir','/content/unused_wbt','--dataset',dataset['dataset'],'--model-name',protocol['backbone']['model_name'],'--data-augmentation','vit','--seed','2025','--num-classes',str(dataset['num_classes']),'--num-tasks',str(dataset['num_tasks']),'--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print('TRAIN FEATURE EXTRACTION START. One progress line appears per task.',flush=True)
    subprocess.run(command,check=True)
assert (cache/'metadata.json').is_file() and (cache/'train.pt').is_file()
assert not (cache/'test.pt').exists(),'FAIL: test.pt became visible'
train=torch.load(cache/'train.pt',weights_only=True,map_location='cpu')
assert tuple(train['features'].shape)==(50000,768) and tuple(train['labels'].shape)==(50000,)
assert torch.isfinite(train['features']).all() and len(torch.unique(train['labels']))==100
print('TRAIN CACHE READY:',tuple(train['features'].shape),'| test.pt absent')
del train

## Locked Phase 1D run

The runner first creates six shared exact-oracle references, then runs 12 inner boundary candidates and 15 outer control units. `START/DONE` marks resumable units; each `TASK` line reports seen-class validation accuracy plus current G/Q error. A long task line means the analytic update is still running.

In [ ]:
# Start/resume Phase 1D. No held-out feature or label is opened.
command=[sys.executable,'-u',RUNNER,'--config',CONFIG,'--dataset-key',DATASET_KEY,'--feature-cache-dir',str(cache),'--output-root',OUTPUT_ROOT,'--device','cuda']
print('STARTING PHASE 1D: 6 references + 12 inner + 15 outer units.',flush=True)
started=time.time(); completed=subprocess.run(command)
print(f'elapsed={(time.time()-started)/60:.1f} minutes | return_code={completed.returncode}',flush=True)
assert completed.returncode==0,'Runner failed; return the complete traceback without changing the protocol.'
RESULT_PATH=Path(OUTPUT_ROOT)/DATASET_KEY/'phase1d_results.json'
assert RESULT_PATH.is_file()
print('PHASE 1D PROCESS: COMPLETE')

In [ ]:
# Compact scientific result: accuracy, G/Q fidelity, low-margin behavior and state.
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
payload=json.loads(RESULT_PATH.read_text())
selection=pd.DataFrame([{'boundary_fraction':x['candidate']['boundary_fraction'],'boundary_strength':x['candidate']['boundary_strength'],'inner_AIA':x['mean_inner_aia'],'inner_stat_error':x['mean_inner_stat_error']} for x in payload['inner_boundary_selection']])
display(selection.sort_values('inner_AIA',ascending=False)); print('selected:',payload['selected_boundary'])
summary=pd.DataFrame([{'method':method,**values} for method,values in payload['outer_summary'].items()]).sort_values('average_incremental_accuracy',ascending=False)
display(summary); print('gates:',json.dumps(payload['gates'],indent=2))
fig,axes=plt.subplots(2,2,figsize=(15,9))
sns.barplot(data=summary,x='method',y='average_incremental_accuracy',ax=axes[0,0]); axes[0,0].set_title('Outer train-validation AIA')
sns.barplot(data=summary,x='method',y='mean_combined_stat_error',ax=axes[0,1]); axes[0,1].set_title('Hard-WTA G/Q error (lower is better)')
sns.barplot(data=summary,x='method',y='low_margin_average_incremental_accuracy',ax=axes[1,0]); axes[1,0].set_title('Lowest Top-K-gap quartile AIA')
sns.barplot(data=summary,x='method',y='persistent_state_bytes',ax=axes[1,1]); axes[1,1].set_title('Persistent learner tensor bytes')
for ax in axes.flat: ax.tick_params(axis='x',rotation=35)
plt.tight_layout(); plt.show()
print('DECISION:',payload['status'],'| uses_test_set=',payload['uses_test_set'])

In [ ]:
# Export evidence. Large oracle G/Q cache tensors and frozen features are excluded.
from google.colab import files
evidence=Path('/content/wbt_soho_phase1d_evidence')
if evidence.exists(): shutil.rmtree(evidence)
shutil.copytree(Path(OUTPUT_ROOT)/DATASET_KEY,evidence,ignore=shutil.ignore_patterns('*.pt'))
for source_name,target_name in [(CONFIG,'locked_config.json'),(RUNNER,'locked_runner.py'),('methods/wbt_soho/transport.py','transport.py'),('methods/wbt_soho/learner.py','learner.py')]: shutil.copy2(source_name,evidence/target_name)
archive=shutil.make_archive('/content/wbt_soho_phase1d_cifar100_train_only','zip',root_dir=evidence)
print('artifact:',archive,'bytes=',Path(archive).stat().st_size,'sha256=',sha(archive))
files.download(archive)